In [0]:
# Design an ETL Pipeline to perform the following:
# 1) Extract data from both of the sources.
# 2) Clean customer names to remove special characters.
# 3)  Replace NULL Amounts with 0
# 4) Data Validation - Order date not in future, customer id in transaction also needs to be present in CRM data too, Amount shall be non-null
# 5) Log unmatched CustomerIDs from Orders to an error table.
# 6) Calculate running total of order amounts per customer and load in a separate table
# 7) Log audit info: row counts before and after each transformation.

In [0]:
# XML
# OrderID	CustomerID	OrderDate	Amount
# O1001	C001	10-01-2023	250
# O1002	C002	12-01-2023	300
# O1003	C004	15-01-2023	NULL

# CSV
# CustomerID	Name	Country
# C001	John@Doe	USA
# C002	Jane#Smith	UK
# C003	Mike!Johnson	Ireland

orders_data = [
    ("O1001", "C001", "10-01-2023", 250.0),
    ("O1002", "C002", "12-01-2023", 300.0),
    ("O1003", "C004", "15-01-2023", None)
]

crm_data = [
    ("C001", "John@Doe", "USA"),
    ("C002", "Jane#Smith", "UK"),
    ("C003", "Mike!Johnson", "Ireland")
]

orders_df = spark.createDataFrame(orders_data, ["OrderID", "CustomerID", "OrderDate", "Amount"])
crm_df = spark.createDataFrame(crm_data, ["CustomerID", "Name", "Country"])

audit_rows = []

def audit(step, before_df, after_df):
    before_count = before_df.count() if before_df is not None else 0
    after_count = after_df.count()
    audit_rows.append((step, before_count, after_count))

In [0]:
# 1) Extract data from both sources
orders_df = spark.createDataFrame(orders_data, ["OrderID", "CustomerID", "OrderDate", "Amount"])
crm_df = spark.createDataFrame(crm_data, ["CustomerID", "Name", "Country"])
audit("extract_orders", None, orders_df)
audit("extract_crm", None, crm_df)


In [0]:
from pyspark.sql.functions import col, regexp_replace, coalesce, lit, to_date, trim,current_date, broadcast, sum as _sum
from pyspark.storagelevel import StorageLevel


# 2) Clean customer names to remove special characters
crm_clean = crm_df.withColumn(
    "Name",
    trim(
        regexp_replace(
            regexp_replace(col("Name"), r"(^[^A-Za-z0-9]+|[^A-Za-z0-9]+$)", ""),
            r"[^A-Za-z0-9]+",
            " "
        )
    )
)

audit("clean_customer_names", crm_df, crm_clean)

crm_clean.show()

# 2) optimized one
# crm_clean = crm_df.withColumn(
#     "Name",
#     regexp_replace(col("Name"), r"[^A-Za-z0-9 ]", "")
# ).persist(StorageLevel.MEMORY_ONLY)



+----------+------------+-------+
|CustomerID|        Name|Country|
+----------+------------+-------+
|      C001|    John Doe|    USA|
|      C002|  Jane Smith|     UK|
|      C003|Mike Johnson|Ireland|
+----------+------------+-------+



In [0]:

# 3) Replace NULL Amounts with 0
orders_clean = orders_df.withColumn("Amount", coalesce(col("Amount"), lit(0.0)))
audit("replace_null_amount", orders_df, orders_clean)

orders_clean.show()


# orders_clean = orders_df.withColumn("Amount", coalesce(col("Amount"), lit(0.0)))
# audit("replace_null_amount", orders_df, orders_clean).persist(StorageLevel.MEMORY_ONLY)

+-------+----------+----------+------+
|OrderID|CustomerID| OrderDate|Amount|
+-------+----------+----------+------+
|  O1001|      C001|10-01-2023| 250.0|
|  O1002|      C002|12-01-2023| 300.0|
|  O1003|      C004|15-01-2023|   0.0|
+-------+----------+----------+------+



In [0]:
# 4) Data validation
orders_typed = orders_df.withColumn("OrderDate", to_date(col("OrderDate"), "dd-MM-yyyy"))
orders_clean = orders_typed.withColumn("Amount", coalesce(col("Amount"), lit(0.0)))
crm_ids = crm_clean.select(col("CustomerID").alias("crm_customer_id"))

print("original")
orders_joined = orders_typed.join(
    broadcast(crm_ids),
    orders_typed.CustomerID == crm_ids.crm_customer_id,
    "left"
)

orders_joined.show()

audit("join_orders_with_crm", orders_typed, orders_joined)

print("valid")
valid_orders = orders_joined.filter(
    (col("OrderDate") <= current_date()) &
    col("crm_customer_id").isNotNull() &
    col("Amount").isNotNull()
).select("OrderID", "CustomerID", "OrderDate", "Amount")

valid_orders.show()

audit("validate_orders", orders_joined, valid_orders)


original
+-------+----------+----------+------+---------------+
|OrderID|CustomerID| OrderDate|Amount|crm_customer_id|
+-------+----------+----------+------+---------------+
|  O1001|      C001|2023-01-10| 250.0|           C001|
|  O1002|      C002|2023-01-12| 300.0|           C002|
|  O1003|      C004|2023-01-15|  NULL|           NULL|
+-------+----------+----------+------+---------------+

valid
+-------+----------+----------+------+
|OrderID|CustomerID| OrderDate|Amount|
+-------+----------+----------+------+
|  O1001|      C001|2023-01-10| 250.0|
|  O1002|      C002|2023-01-12| 300.0|
+-------+----------+----------+------+



In [0]:
# 5) Log unmatched CustomerIDs from Orders to an error table
error_table = orders_joined.filter(
    col("crm_customer_id").isNull()
).select("OrderID", "CustomerID", "OrderDate", "Amount")

error_table.show()

audit("log_unmatched_customers", None, error_table)


+-------+----------+----------+------+
|OrderID|CustomerID| OrderDate|Amount|
+-------+----------+----------+------+
|  O1003|      C004|2023-01-15|   0.0|
+-------+----------+----------+------+



In [0]:
# 6) Calculate running total of order amounts per customer
window_spec = Window.partitionBy("CustomerID").orderBy("OrderDate", "OrderID").rowsBetween(Window.unboundedPreceding, Window.currentRow)

running_total_table = valid_orders.withColumn(
    "running_total_amount",
    _sum("Amount").over(window_spec)
)

running_total_table.show()
audit("calculate_running_total", valid_orders, running_total_table)



+-------+----------+----------+------+--------------------+
|OrderID|CustomerID| OrderDate|Amount|running_total_amount|
+-------+----------+----------+------+--------------------+
|  O1001|      C001|2023-01-10| 250.0|               250.0|
|  O1002|      C002|2023-01-12| 300.0|               300.0|
+-------+----------+----------+------+--------------------+



In [0]:
# 7) Log audit info
audit_table = spark.createDataFrame(audit_rows, ["step_name", "before_count", "after_count"])

print("step 0")
orders_df.show()
crm_df.show()
print("step 1")
crm_clean.show()
print("step 2")
orders_clean.show()
print("step 3")
valid_orders.show()
print("step 5")
error_table.show()
print("step 6")
running_total_table.show()
print("step 7")
audit_table.show()

step 0
+-------+----------+----------+------+
|OrderID|CustomerID| OrderDate|Amount|
+-------+----------+----------+------+
|  O1001|      C001|10-01-2023| 250.0|
|  O1002|      C002|12-01-2023| 300.0|
|  O1003|      C004|15-01-2023|  NULL|
+-------+----------+----------+------+

+----------+------------+-------+
|CustomerID|        Name|Country|
+----------+------------+-------+
|      C001|    John@Doe|    USA|
|      C002|  Jane#Smith|     UK|
|      C003|Mike!Johnson|Ireland|
+----------+------------+-------+

step 1
+----------+------------+-------+
|CustomerID|        Name|Country|
+----------+------------+-------+
|      C001|    John Doe|    USA|
|      C002|  Jane Smith|     UK|
|      C003|Mike Johnson|Ireland|
+----------+------------+-------+

step 2
+-------+----------+----------+------+
|OrderID|CustomerID| OrderDate|Amount|
+-------+----------+----------+------+
|  O1001|      C001|2023-01-10| 250.0|
|  O1002|      C002|2023-01-12| 300.0|
|  O1003|      C004|2023-01-15| 

In [0]:
# Design an ETL Pipeline to perform the following:
# 1) Extract data from both of the sources.
# 2) Clean customer names to remove special characters.
# 3)  Replace NULL Amounts with 0
# 4) Data Validation - Order date not in future, customer id in transaction also needs to be present in CRM data too, Amount shall be non-null
# 5) Log unmatched CustomerIDs from Orders to an error table.
# 6) Calculate running total of order amounts per customer and load in a separate table
# 7) Log audit info: row counts before and after each transformation.